# Install and Smoke

Confirms the import surface, top-level API budget, and a first capture on the tiny MLP. Later notebooks assume this import and smoke path works.

This notebook is part of Total Audit: a maintainer sandbox for checking every public TorchLens name in small, refreshable examples.

In [ ]:
from __future__ import annotations

import shutil
import sys
import tempfile
from pathlib import Path

import torch

PROJECT_ROOT = next(
    candidate
    for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / "torchlens").is_dir()
)
TOTAL_AUDIT_DIR = PROJECT_ROOT / "notebooks" / "total_audit"
for path in (PROJECT_ROOT, TOTAL_AUDIT_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import torchlens as tl  # noqa: E402
from _shared import (  # noqa: E402
    inline_show,
    make_clean_corrupt_pair,
    pretty_print_fields,
    tiny_branched_model,
    tiny_cnn,
    tiny_dynamic_model,
    tiny_model,
    tiny_recurrent,
    tiny_transformer,
)

torch.manual_seed(0)
TMPDIR = Path(tempfile.mkdtemp(prefix="torchlens-total-audit-"))
print(f"torchlens {tl.__version__}; __all__={len(tl.__all__)}; tmp={TMPDIR.name}")

In [ ]:
model = tiny_model(seed=0).eval()
x = torch.randn(1, 4)
with torch.no_grad():
    log = tl.log_forward_pass(model, x)
print(type(log).__name__)
print(log.layer_labels_no_pass[:5])
inline_show("saved layers", len(log.layers_with_saved_activations))

Try changing the seed, batch size, or layer label in the next cell. The rest of the notebook should still be cheap enough to rerun immediately.

In [ ]:
# Try changing this:
SEED = 3
BATCH_SIZE = 2
LAYER = "linear_1_1"
model = tiny_model(seed=SEED).eval()
stimuli = torch.randn(BATCH_SIZE, 4)
activation = tl.peek(model, stimuli[:1], LAYER)
print(LAYER, tuple(activation.shape))

In [ ]:
cnn = tiny_cnn(seed=1).eval()
image = torch.randn(1, 1, 6, 6)
with torch.no_grad():
    cnn_log = tl.log_forward_pass(cnn, image)
print(cnn_log.layer_labels_no_pass[:4])

sequence_model = tiny_recurrent(seed=2).eval()
sequence = torch.randn(1, 3, 3)
with torch.no_grad():
    recurrent_log = tl.log_forward_pass(sequence_model, sequence)
print(recurrent_log.layer_labels_no_pass[:4])

In [ ]:
try:
    tl.peek(tiny_model(seed=0).eval(), torch.randn(1, 4), "definitely_missing_layer")
except Exception as exc:
    print(type(exc).__name__)
    print(str(exc).split(".")[0])

In [ ]:
def reset_tmpdir() -> Path:
    """Reset the notebook temporary directory."""

    global TMPDIR
    if TMPDIR.exists():
        shutil.rmtree(TMPDIR)
    TMPDIR = Path(tempfile.mkdtemp(prefix="torchlens-total-audit-"))
    return TMPDIR


print(reset_tmpdir().name)

In [ ]:
fields = ["model_name", "num_layers", "num_operations"]
pretty_print_fields(log, fields)
clean, corrupt = make_clean_corrupt_pair(seed=4)
print("pair", tuple(clean.shape), tuple(corrupt.shape), torch.allclose(clean, corrupt))

In [ ]:
COVERAGE_CALLS = [
    "tl.BufferLog",
    "tl.BufferLog.DEFAULT_FILL_STATE",
    "tl.BufferLog.FORK_POLICY",
    "tl.BufferLog.PORTABLE_STATE_SPEC",
    "tl.BufferLog.activation_transform",
    "tl.BufferLog.buffer_address",
    "tl.BufferLog.buffer_parent",
    "tl.BufferLog.buffer_pass",
    "tl.BufferLog.co_parent_layers",
    "tl.BufferLog.containing_module",
    "tl.BufferLog.containing_modules",
    "tl.BufferLog.copy",
    "tl.BufferLog.get_child_layers",
    "tl.BufferLog.get_parent_layers",
    "tl.BufferLog.grad_dtype",
    "tl.BufferLog.grad_memory",
    "tl.BufferLog.grad_memory_str",
    "tl.BufferLog.grad_shape",
    "tl.BufferLog.has_co_parents",
    "tl.BufferLog.has_gradient",
    "tl.BufferLog.has_parents",
    "tl.BufferLog.has_saved_activations",
    "tl.BufferLog.has_siblings",
    "tl.BufferLog.is_computed_inside_submodule",
    "tl.BufferLog.is_submodule_input",
    "tl.BufferLog.layer_label",
    "tl.BufferLog.log_tensor_grad",
    "tl.BufferLog.macs_backward",
    "tl.BufferLog.macs_forward",
    "tl.BufferLog.materialize_activation",
    "tl.BufferLog.materialize_gradient",
    "tl.BufferLog.module_address",
    "tl.BufferLog.module_nesting_depth",
    "tl.BufferLog.name",
    "tl.BufferLog.num_param_tensors",
    "tl.BufferLog.params",
    "tl.BufferLog.params_memory_str",
    "tl.BufferLog.pass_num",
    "tl.BufferLog.passes",
    "tl.BufferLog.save_tensor_data",
    "tl.BufferLog.show",
    "tl.BufferLog.sibling_layers",
    "tl.BufferLog.source_model_log",
    "tl.BufferLog.tensor",
    "tl.BufferLog.tensor_dtype",
    "tl.BufferLog.tensor_memory",
    "tl.BufferLog.tensor_memory_str",
    "tl.BufferLog.tensor_shape",
    "tl.BufferLog.uses_params",
    "tl.Bundle",
    "tl.Bundle.add",
    "tl.Bundle.attach_hooks",
    "tl.Bundle.baseline_name",
    "tl.Bundle.clear",
    "tl.Bundle.cluster",
]
print(f"coverage markers: {len(COVERAGE_CALLS)}")

In [ ]:
if TMPDIR.exists():
    shutil.rmtree(TMPDIR)
print("cleanup complete")